# Installation

In [1]:
%pip install -q pytz

# Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ── Edit this to match where your TOX folder lives in Drive ──────
DRIVE_ROOT = '/content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX'
os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results_tox21',     exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)
print('Contents:', os.listdir(DRIVE_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Root: /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX
Contents: ['tox21_analysis.ipynb', 'tox21_dataset (1).csv', 'tox21_dataset.csv', 'TOX3GNN.py', 'optim_TOX3GNN.py', 'results_tox21', 'feat_corr.py', '__pycache__', 'imputation_checkpoints', 'tox21_imputed.csv', 'checkpoints_tox21', 'TOX3GNN_experiment.ipynb', 'utils.py', 'TOX3GNN_multitask.ipynb']


In [5]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'optuna'],  check=True)

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('Dependencies ready.')


torch=2.11.0+cpu  cuda=False
Dependencies ready.


In [6]:
sys.path.insert(0, DRIVE_ROOT)

from utils import (
    one_hot_encoding,
    get_atom_features,
    get_bond_features,
    smiles_to_graph_list,
    scaffold_split,
    save_ckp,
    load_ckp,
    optimizer_to,
    round_to_4,
)

# Reading the Data

In [ ]:
df=pd.read_csv('tox21_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'tox21_dataset.csv'

In [ ]:
df.head()

NameError: name 'df' is not defined

In [ ]:
task_columns = [col for col in df.columns if col not in ['smiles', 'mol_id']]
df[['smiles', 'mol_id']] = df[['smiles', 'mol_id']].astype('string')
# cast all the task columns to 'Int64' type to handle NaN values
df[task_columns] = df[task_columns].apply(pd.to_numeric, errors='coerce').astype('Int64')
df.info()

# Null Values

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
# excluded mol_id and smiles
null_values =df.isna().sum().sort_values(ascending=True)[2:]
col_names =df.columns

null_bars, ax = plt.subplots(figsize=(10, 7))

threshold = 1000
total_rows = df.shape[0]

# Colors based on threshold
colors = [
    'purple' if v < threshold else 'lightblue'
    for v in null_values.values
]
bars = ax.bar(
    null_values.index,
    null_values.values,
    color=colors
)
labels = [
    f'{v / total_rows * 100:.2f}%'
    for v in null_values.values
]

ax.bar_label(
    bars,
    labels=labels,
    padding=3,
    fontsize=10
)
ax.axhline(
    y=threshold,
    color='red',
    linestyle='--',
    label=f'Threshold ({threshold})'
)

ax.set_title(
    'Number of Null Values per Task',
    fontsize=14,
    pad=15
)
ax.set_ylabel('Count of Null Values')
ax.set_xlabel('Tasks')
plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha='right'
)
ax.grid(False)
ax.legend()
null_bars.tight_layout()
plt.show()

In [ ]:
df.head()

NameError: name 'df' is not defined

# Toxic Compounds in All Tasks

## Pie Plots

In [ ]:
# visualizing the toxic compounds per Task (which are all except the mol_id and smiles columns)
value_counts_dict = {task: df[task].value_counts() for task in task_columns}

from matplotlib.patches import Patch

pie_plot, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, task in enumerate(task_columns):
    ax = axes[i]

    value_counts_dict[task].plot.pie(
        ax=ax,
        autopct='%.2f',
        colors=["#087134", 'lightcoral'],
        labels=None  # Remove labels next to slices
    )

    ax.set_title(task)
    ax.set_ylabel('')

legend_handles = [
    Patch(facecolor='lightblue', label='Non-Toxic'),
    Patch(facecolor='#055526', label='Toxic')
]

pie_plot.legend(
    handles=legend_handles,
    loc='upper right',
    fontsize=10
)

pie_plot.tight_layout()
plt.show()

In [ ]:
tox_ntox_line, ax = plt.subplots(figsize=(12, 5))

# Count toxic and non-toxic compounds
toxic_df = pd.DataFrame({
    'toxic': (df[task_columns] == 1).sum(),
    'non_toxic': (df[task_columns] == 0).sum()
})

toxic_df_sorted = toxic_df.sort_values(by='toxic')

colors = {
    'toxic': "lightcoral",
    'non_toxic': "#055526"
}

toxic_df_sorted.plot(
    kind='line',
    ax=ax,
    color=[colors['toxic'], colors['non_toxic']],
    alpha=0.8
)

for line in ax.get_lines():
    line.set_marker('o')

ax.set_xticks(np.arange(len(toxic_df_sorted)))
ax.set_xticklabels(
    toxic_df_sorted.index,
    fontsize=8,
    rotation=45,
    ha='right'
)
ax.legend(['Toxic', 'Non-Toxic'], loc='upper right', fontsize=10)
ax.set_title('# Toxic vs Non-Toxic per Task', fontsize=12)
ax.set_ylabel('Compound Count', fontsize=10)
ax.set_xlabel('Tox21 Tasks', fontsize=6)
ax.grid(axis='y', linestyle='--', alpha=0.5)

tox_ntox_line.tight_layout()
plt.show()

In [ ]:
import rdkit.Chem as Chem
from rdkit.Chem import MolFromSmiles
from  rdkit.Chem.Scaffolds import MurckoScaffold
import random
def get_scaffold(smiles:str)->str:
    """
    Given a SMILES string, return its Murcko scaffold as a SMILES string.
    """
    mol = MolFromSmiles(smiles)
    if not mol:
        return ""
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)

def scaffold_split(smiles_list, train_frac=0.6, val_frac=0.2, seed=42):
    scaffold_idx_dict = defaultdict(list)
    for idx,smi in enumerate(smiles_list):
        scaffold = get_scaffold(smi)
        scaffold_idx_dict[scaffold].append(idx)
    scaffold_groups =list(scaffold_idx_dict.values())
    rng = random.Random(seed)
    rng.shuffle(scaffold_groups)

    n_total = len(smiles_list)
    train_cutoff = int(train_frac * n_total)
    val_cutoff   = int((train_frac + val_frac) * n_total)
    train_idx, val_idx, test_idx = [], [], []
    for group in scaffold_groups:
        if len(train_idx) < train_cutoff:
            train_idx.extend(group)
        elif len(train_idx) + len(val_idx) < val_cutoff:
            val_idx.extend(group)
        else:
            test_idx.extend(group)
    print(f"Scaffold split → train: {len(train_idx)}, "
          f"val: {len(val_idx)}, test: {len(test_idx)}")
    return train_idx, val_idx, test_idx, scaffold_idx_dict


ModuleNotFoundError: No module named 'rdkit'

In [ ]:
from collections import defaultdict

train_idx, val_idx, test_idx, scaffold_idx_dict = scaffold_split(list(df["smiles"]))


In [ ]:
index_to_scaffold = {
    idx: scaf
    for scaf, indices in scaffold_idx_dict.items()
    for idx in indices
}


In [ ]:
df_scaffolds = df.drop(columns = 'mol_id').copy()
df_scaffolds['scaffolds'] = df_scaffolds.index.map(index_to_scaffold)
df_scaffolds['scaffolds']  = df_scaffolds['scaffolds'].astype('string')
df_scaffolds.head()

In [ ]:
def smiles_to_mol(smile:str):
    if isinstance(smile, str) and smile:
        return Chem.MolFromSmiles(smile)
    return None

In [ ]:
df_mol_scaffold =df_scaffolds[['smiles', 'scaffolds']].copy()
df_mol_scaffold['mol_smiles'] = df_mol_scaffold['smiles'].apply(smiles_to_mol)
df_mol_scaffold['mol_scaffold'] = df_mol_scaffold['scaffolds'].apply(smiles_to_mol)
df_mol_scaffold = df_mol_scaffold.astype({'smiles': 'string', 'scaffolds': 'string'})

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import HTML

def mol_to_svg(mol, width=200, height=150):
    """Converts RDKit Mol object to an inline HTML SVG string."""
    if mol is None:
        return ""
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return drawer.GetDrawingText()

In [ ]:
df_mol_scaffold['img_smiles'] = df_mol_scaffold['mol_smiles'].apply(mol_to_svg)
df_mol_scaffold['img_scaffold'] = df_mol_scaffold['mol_scaffold'].apply(mol_to_svg)

In [ ]:
HTML(df_mol_scaffold[['smiles', 'img_smiles', 'scaffolds', 'img_scaffold']].head().to_html(escape=False))

In [ ]:
print(f"Unique Compounds: {df_mol_scaffold['smiles'].nunique()}" + "\n"+ \
     f"Unique Scaffolds: {df_mol_scaffold['scaffolds'].nunique()}")

Find out which and how many of those scaffolds are toxic

In [ ]:
df_mol_scaffold.info()

In [ ]:
df_mol_scaffold['scaffolds'].value_counts().head(10)

NameError: name 'df_mol_scaffold' is not defined

In [ ]:
len(df_mol_scaffold[df_mol_scaffold['scaffolds']==''])

In [ ]:
import random
from collections import defaultdict
from typing import Tuple, List, Union, Optional
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def _smiles_to_scaffold(smi: str) -> str:
    """Helper for parallel execution: Convert single SMILES to Murcko Scaffold SMILES."""
    if not isinstance(smi, str) or not smi:
        return ""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return ""


def get_scaffolds_parallel(
    smiles_series: pd.Series,
    n_jobs: int = -1
) -> pd.Series:
    """
    Computes Murcko scaffolds for a Pandas Series of SMILES using parallel processing.
    """
    import os
    if n_jobs == -1:
        n_jobs = max(1, os.cpu_count() - 1)

    smiles_list = smiles_series.tolist()

    if n_jobs > 1:
        with ProcessPoolExecutor(max_workers=n_jobs) as executor:
            scaffolds = list(executor.map(_smiles_to_scaffold, smiles_list, chunksize=500))
    else:
        scaffolds = [_smiles_to_scaffold(s) for s in smiles_list]

    return pd.Series(scaffolds, index=smiles_series.index, name="scaffold")


def scaffold_split(
    data: Union[pd.DataFrame, pd.Series, List[str]],
    smiles_col: Optional[str] = "smiles",
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    test_frac: float = 0.1,
    seed: int = 42,
    n_jobs: int = -1,
    balanced: bool = True
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    High-performance, Pythonic Bemis-Murcko scaffold splitter for pandas DataFrames/Series.

    Parameters
    ----------
    data : pd.DataFrame, pd.Series, or list
        Dataset containing SMILES strings.
    smiles_col : str, optional
        Name of column containing SMILES (if data is a DataFrame).
    train_frac : float
        Fraction of data for training.
    val_frac : float
        Fraction of data for validation.
    test_frac : float
        Fraction of data for testing.
    seed : int
        Random seed for shuffling equal-sized scaffold groups.
    n_jobs : int
        Number of CPU workers for parallel RDKit scaffold parsing (-1 uses all cores).
    balanced : bool
        If True, sorts scaffold clusters by size descending (Greedy Bin Packing),
        ensuring optimal split sizes.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        Arrays of train, validation, and test indices.
    """
    assert np.isclose(train_frac + val_frac + test_frac, 1.0), "Splits must sum to 1.0"

    # Extract Series of SMILES
    if isinstance(data, pd.DataFrame):
        smiles_series = data[smiles_col]
    elif isinstance(data, list):
        smiles_series = pd.Series(data)
    else:
        smiles_series = data

    # 1. Parallel computation of scaffolds
    scaffolds = get_scaffolds_parallel(smiles_series, n_jobs=n_jobs)

    # 2. Group indices by scaffold using Pandas groupby (C-speed grouping)
    scaffold_to_indices = defaultdict(list)
    for idx, scaf in zip(smiles_series.index, scaffolds):
        scaffold_to_indices[scaf].append(idx)

    # 3. Sort scaffold clusters
    # Sorting by cluster size descending (Greedy Bin Packing) guarantees better split ratios
    if balanced:
        # Sort by size descending, then shuffle groups of identical size deterministically
        rng = random.Random(seed)
        scaffold_groups = list(scaffold_to_indices.values())
        rng.shuffle(scaffold_groups)
        scaffold_groups.sort(key=len, reverse=True)
    else:
        scaffold_groups = list(scaffold_to_indices.values())
        rng = random.Random(seed)
        rng.shuffle(scaffold_groups)

    # 4. Greedy assignment to train / val / test sets
    n_total = len(smiles_series)
    train_cutoff = train_frac * n_total
    val_cutoff = (train_frac + val_frac) * n_total

    train_idx, val_idx, test_idx = [], [], []

    for group in scaffold_groups:
        if len(train_idx) + len(group) <= train_cutoff:
            train_idx.extend(group)
        elif len(train_idx) + len(val_idx) + len(group) <= val_cutoff:
            val_idx.extend(group)
        else:
            test_idx.extend(group)

    print(f"Scaffold Split Completed ({n_total} total compounds):")
    print(f"  Train : {len(train_idx):>6d} ({len(train_idx)/n_total:.2%})")
    print(f"  Val   : {len(val_idx):>6d} ({len(val_idx)/n_total:.2%})")
    print(f"  Test  : {len(test_idx):>6d} ({len(test_idx)/n_total:.2%})")

    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

In [ ]:
df['smiles'].describe()['top']

In [ ]:
# TODO: chain the functions
value_counts_df = pd.DataFrame(value_counts_dict).T
value_counts_df.rename(columns={0: 'Non-Toxic', 1: 'Toxic'}, inplace=True)

value_counts_df['ratio'] = np.round(value_counts_df['Toxic'] / (value_counts_df['Toxic'] + value_counts_df['Non-Toxic']), 2)
value_counts_df.sort_values(by='ratio', ascending=False, inplace=True)
value_counts_df.head()


In [ ]:
# identify valid smiles
valid_mask = df['smiles'].apply(lambda s: Chem.MolFromSmiles(s) is not None)

In [ ]:
for k,v in value_counts_dict.items():
    print(f"Task: {k}, Non-Toxic: {v.get(0, 0)}, Toxic: {v.get(1, 0)}")

In [ ]:
null_values = pd.DataFrame(null_values, columns=['Null Count'])
null_values['null_percentage'] = np.round((null_values['Null Count'] / len(df)) * 100, 2)
null_values.head()

In [ ]:
# merge null_values and value_counts_df based on the index (task names)
merged_summary = pd.merge(null_values, value_counts_df, left_index=True, right_index=True)
merged_summary.sort_values(by='Null Count', ascending=False, inplace=True)
merged_summary.head()

# Relationship of Null Values and Toxic Compounds per Task

In [ ]:
plt.figure(figsize=(12, 6))
# plot bars null and toxic ratio on the same plot with different colors and a legend
plt.bar(merged_summary.index, merged_summary['null_percentage'],  color='orange', alpha=0.5, label='Null Values', )
plt.bar(merged_summary.index, merged_summary['ratio']*100, color='blue', alpha=0.5, label='Toxic Compounds Ratio')
plt.xlabel('Tasks')
plt.ylabel('Percentage %')
plt.title('Relationship of Null Values and Toxic Compounds per Task')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Based on the visualization we see no consisteny but we should compare the results with the prediction-imputed version

Questions?
## Feature Correlation
Which features are more important for toxicity prediction?
(feature correlation, PCA)
Which column has the largest sum of toxicity

## Fragmentation
How to identify the toxic fragments per column

## Toxicity
Run pIC50 , etc tests to filter out the molecules
that are inherently toxic ( remove them if they are
the minority)
Filters
Ashford/Brenk alerts or REOS filters (for general reactivity/toxicity).

Pain/Pan-Assay Interference Compounds (PAINS): Identifies molecules that falsely interfere with assay readouts rather than causing real biological toxicity.